## Example 1: The breaker opens after too many failures

In [ ]:
import pybreaker

# Example 1: after fail_max failures in a row the breaker opens and refuses calls at once
breaker = pybreaker.CircuitBreaker(fail_max=2, reset_timeout=60)
calls = []


@breaker
def flaky():
    calls.append(1)
    raise ConnectionError("service down")


for _ in range(3):
    try:
        flaky()
    except ConnectionError:
        assert breaker.current_state == pybreaker.STATE_CLOSED or breaker.fail_counter == 2
    except pybreaker.CircuitBreakerError:
        pass
assert breaker.current_state == pybreaker.STATE_OPEN and breaker.fail_counter == 2
assert len(calls) == 2, calls
print(f"Example 1 OK: state = {breaker.current_state}, the function ran {len(calls)} of 3 times")

## Example 2: After the reset timeout one trial call closes it again

In [ ]:
import time

import pybreaker

# Example 2: once reset_timeout has passed the breaker is half-open; a successful call closes it
breaker = pybreaker.CircuitBreaker(fail_max=1, reset_timeout=0.2)
state = {"healthy": False}


def call_service():
    if not state["healthy"]:
        raise TimeoutError("still down")
    return "ok"


try:
    breaker.call(call_service)
except (TimeoutError, pybreaker.CircuitBreakerError):
    pass
assert breaker.current_state == pybreaker.STATE_OPEN
time.sleep(0.3)
state["healthy"] = True
assert breaker.call(call_service) == "ok"
assert breaker.current_state == pybreaker.STATE_CLOSED and breaker.fail_counter == 0
print("Example 2 OK: open -> half-open -> closed after one successful trial call")

## Example 3: Listeners see every state change and every failure

In [ ]:
import pybreaker

# Example 3: a CircuitBreakerListener receives callbacks; here it records the transitions
class Recorder(pybreaker.CircuitBreakerListener):
    def __init__(self):
        self.changes, self.failures = [], 0

    def state_change(self, cb, old_state, new_state):
        self.changes.append((old_state.name, new_state.name))

    def failure(self, cb, exc):
        self.failures += 1


recorder = Recorder()
breaker = pybreaker.CircuitBreaker(fail_max=3, listeners=[recorder], name="payments")
for _ in range(3):
    try:
        breaker.call(lambda: 1 / 0)
    except (ZeroDivisionError, pybreaker.CircuitBreakerError):
        pass
assert recorder.failures == 3 and recorder.changes == [("closed", "open")], (recorder.failures, recorder.changes)
assert breaker.name == "payments"
print(f"Example 3 OK: {recorder.failures} failures seen, transitions {recorder.changes}")

## Example 4: Excluded exceptions are not counted as failures

In [ ]:
import pybreaker

# Example 4: business errors (here a KeyError) pass through without tripping the breaker
breaker = pybreaker.CircuitBreaker(fail_max=2, exclude=[KeyError])
lookup = {"a": 1}
for _ in range(5):
    try:
        breaker.call(lambda: lookup["missing"])
    except KeyError:
        pass
assert breaker.fail_counter == 0 and breaker.current_state == pybreaker.STATE_CLOSED
breaker.close()
for _ in range(2):
    try:
        breaker.call(lambda: lookup["a"] / 0)
    except (ZeroDivisionError, pybreaker.CircuitBreakerError):
        pass
assert breaker.current_state == pybreaker.STATE_OPEN
print("Example 4 OK: 5 excluded KeyErrors left it closed; 2 real failures opened it")

## Example 5: redis's multi-database client builds its circuit breaker on pybreaker

In [ ]:
import pybreaker
from redis.multidb.circuit import PBCircuitBreakerAdapter, State

# Example 5: redis.multidb wraps a pybreaker.CircuitBreaker; this module did not import before pybreaker shipped
changes = []
adapter = PBCircuitBreakerAdapter(pybreaker.CircuitBreaker(fail_max=1, reset_timeout=30))
adapter.on_state_changed(lambda cb, old, new: changes.append((old, new)))
assert adapter.state == State.CLOSED
adapter.state = State.OPEN
assert adapter.state == State.OPEN, adapter.state
adapter.state = State.CLOSED
assert adapter.state == State.CLOSED and changes, changes
print(f"Example 5 OK: redis.multidb.circuit drove pybreaker through {len(changes)} state changes")